# Module 8.5 — Conversation and Context Management

Companion notebook to [`docs/modules/08_5_conversation_and_context_management.md`](../docs/modules/08_5_conversation_and_context_management.md).

Like every module since Module 6, this is fully built and tested without a live runtime —
`SessionStore` uses stdlib `sqlite3` directly, and the budget/truncation/summarization
labs run against synthetic conversations and `FakeRuntime`. Only Lab 5 (real recall
measurement) is honest-skip.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "packages"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_01"))
sys.path.insert(0, str(REPO_ROOT / "scripts" / "module_08_5"))
print(f"Repo root: {REPO_ROOT}")

Repo root: /Users/bhakti/workspace/local_ai_app


## 1. SessionStore: real SQLite persistence, proven across a restart

In [2]:
import tempfile
from pathlib import Path as P

from local_ai_core.conversation.session_store import SessionStore
from local_ai_core.conversation.turn import Turn

db_path = P(tempfile.mkdtemp()) / "demo_sessions.db"

store1 = SessionStore(db_path)
store1.append_turn("demo", Turn(role="system", content="You are a helpful assistant.", sticky=True))
store1.append_turn("demo", Turn(role="user", content="My favorite color is teal."))
store1.close()
print("Closed store1 (simulating an app restart)...")

store2 = SessionStore(db_path)  # a genuinely NEW instance, same file
turns = store2.get_turns("demo")
print(f"Turns recovered after restart: {len(turns)}")
for t in turns:
    print(f"  {t.role}: {t.content}")

Closed store1 (simulating an app restart)...
Turns recovered after restart: 2
  system: You are a helpful assistant.
  user: My favorite color is teal.


## 2. Memory deletion (Lab 6)

In [3]:
print(f"Before delete: {len(store2.get_turns('demo'))} turns")
store2.delete_session("demo")
print(f"After delete: {len(store2.get_turns('demo'))} turns")
store2.close()

Before delete: 2 turns
After delete: 0 turns


## 3. Lab 3: forcing a conversation past the context window

In [4]:
import force_past_context_window as lab3

result = lab3.run_lab(n_turns=40, context_window=2000)
print(lab3.result_to_markdown(result))

# Lab 3 — forcing a conversation past the context window

- Original turns: 41 (~2710 tokens)
- History budget: 1500 tokens
- Exceeded budget before truncation: True
- After drop_oldest: 23 turns (~1494 tokens)
- Exceeded budget after truncation: False
- Sticky system prompt retained: True



## 4. Lab 4: drop-oldest vs. summarization buffer — which retains an early fact?

In [5]:
import compare_truncation_strategies as lab4

result4 = lab4.run_lab(n_turns=30, context_window=1500)
print(lab4.result_to_markdown(result4))

# Lab 4 — drop-oldest vs. summarization buffer

Early fact injected: `My account number is ACC-88213.`

| Strategy | Final turn count | Early fact recoverable |
|---|---:|---:|
| drop_oldest | 19 | no |
| summarize_then_truncate | 4 | yes |



This matches the strategy comparison table's tradeoff directly: drop-oldest loses the early
commitment entirely once enough conversation piles up after it, while the summarization
buffer keeps a compressed trace of it (via the injected summarize_fn) even though the raw
turn itself is gone.

## 5. Tool-call/tool-result pairing is never split

In [6]:
from local_ai_core.conversation.token_budget import ConversationBudget, heuristic_token_counter
from local_ai_core.conversation.truncation import drop_oldest

tool_call = Turn(role="assistant", content="calling get_weather(city='Tokyo')", turn_group_id="call-1")
tool_result = Turn(role="tool", content="22C, clear skies", turn_group_id="call-1", tool_call_id="call-1")
conversation = [Turn(role="user", content="old filler " * 20)] * 3 + [tool_call, tool_result]

tight_budget = ConversationBudget(context_window=10, reserved_system=0, reserved_tools=0, reserved_current_user_turn=0, reserved_answer=0)
result5 = drop_oldest(conversation, tight_budget, heuristic_token_counter)

print(f"tool_call in result: {tool_call in result5}")
print(f"tool_result in result: {tool_result in result5}")
print("Both present or both absent (never split):", (tool_call in result5) == (tool_result in result5))

tool_call in result: True
tool_result in result: True
Both present or both absent (never split): True


## 6. A live chat loop turn, against FakeRuntime

In [7]:
import chat_loop
from local_ai_core.runtimes.fake import FakeRuntime

demo_store = SessionStore()
demo_runtime = FakeRuntime(default_response="Nice to meet you! How can I help?")

reply = await chat_loop.process_user_input(demo_store, "chat-demo", "Hi, I'm new here.", demo_runtime, "demo-model", system_prompt="You are a helpful assistant.")
print(f"Assistant: {reply.content}")
print("\nFull session history:")
for t in demo_store.get_turns("chat-demo"):
    print(f"  [{t.role}]{' (sticky)' if t.sticky else ''}: {t.content}")

Assistant: Nice to meet you! How can I help?

Full session history:
  [system] (sticky): You are a helpful assistant.
  [user]: Hi, I'm new here.
  [assistant]: Nice to meet you! How can I help?


## 7. Real recall measurement, if available

Expected to skip on this machine (no local model runtime installed by design). Real recall
measurement (Lab 5) requires asking a real model a question that depends on an early turn
and checking whether it actually answers correctly - not something a fake can honestly show.

In [8]:
from ollama_probe import is_ollama_available

if is_ollama_available():
    from local_ai_core.runtimes.ollama import OllamaRuntime
    real_store = SessionStore()
    real_runtime = OllamaRuntime()
    await chat_loop.process_user_input(real_store, "recall-test", "My account number is ACC-99213. Remember it.", real_runtime, "qwen2.5:1.5b")
    recall_reply = await chat_loop.process_user_input(real_store, "recall-test", "What is my account number?", real_runtime, "qwen2.5:1.5b")
    print(f"Recall answer: {recall_reply.content}")
    await real_runtime.aclose()
else:
    print("SKIPPED: Ollama is not reachable at http://localhost:11434.")
    print("This machine deliberately has no local model runtime installed (see PROGRESS.md).")
    print("On the resourced 32GB Mac: re-run this cell for a real recall measurement.")

SKIPPED: Ollama is not reachable at http://localhost:11434.
This machine deliberately has no local model runtime installed (see PROGRESS.md).
On the resourced 32GB Mac: re-run this cell for a real recall measurement.


## 8. Take it to the command line

```bash
uv run python scripts/module_08_5/force_past_context_window.py
uv run python scripts/module_08_5/compare_truncation_strategies.py
uv run python scripts/module_08_5/chat_loop.py --model qwen2.5:1.5b   # needs Ollama, interactive
```

Then fold the output into
[`reports/module_08_5_conversation_memory_report.md`](../reports/module_08_5_conversation_memory_report.md).